# 02. 노션 문서 기반 RAG 시스템

노션 API를 연동하여 실제 문서를 기반으로 RAG 시스템을 구축합니다.

## 학습 내용
1. 노션 API 연동 설정
2. 노션 데이터베이스에서 문서 로드
3. 벡터 DB 구축 (PostgreSQL + pgvector)
4. RAG 파이프라인 실행 (Anthropic Claude)
5. 대화형 RAG 구현

## 기술 스택
- **LLM**: Anthropic Claude (claude-sonnet-4-20250514)
- **임베딩**: OpenAI text-embedding-3-small (또는 HuggingFace)
- **벡터 DB**: PostgreSQL + pgvector

## 환경 설정

In [ ]:
import os
import sys

# 프로젝트 루트를 path에 추가
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

# 환경 변수 확인
print(f"Anthropic API Key: {'설정됨' if os.getenv('ANTHROPIC_API_KEY') else '미설정'}")
print(f"OpenAI API Key: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정 (HuggingFace 사용)'}")
print(f"Notion API Key: {'설정됨' if os.getenv('NOTION_API_KEY') else '미설정'}")
print(f"PostgreSQL Host: {os.getenv('POSTGRES_HOST', 'localhost')}")
print(f"PostgreSQL Port: {os.getenv('POSTGRES_PORT', '5433')}")

---
## 1. 노션 Integration 설정 가이드

### Step 1: Integration 생성
1. https://www.notion.so/my-integrations 접속
2. "New integration" 클릭
3. 이름 입력 (예: "RAG System")
4. Workspace 선택
5. "Submit" 클릭
6. **Internal Integration Token** 복사 → `.env`의 `NOTION_API_KEY`에 저장

### Step 2: 페이지/데이터베이스에 권한 부여
1. 연동할 노션 페이지 또는 데이터베이스 열기
2. 우측 상단 "..." 클릭 → "Connections" → Integration 선택
3. 방금 생성한 Integration 추가

### Step 3: 데이터베이스 ID 확인
- 데이터베이스 URL 형식: `https://www.notion.so/{workspace}/{database_id}?v=...`
- `database_id` 부분 (32자 hex)을 복사

---
## 2. 샘플 문서로 테스트

노션 연동 전, 샘플 문서로 RAG 파이프라인을 테스트합니다.

In [ ]:
from src.loaders.notion_loader import create_sample_documents
from src.embeddings.embedding_manager import EmbeddingManager
from src.vectorstore.postgres_store import PostgresVectorStore
from src.llm.model_adapter import LLMAdapter
from src.chains.rag_chain import RAGChain

# 샘플 문서 생성
sample_docs = create_sample_documents()
print(f"샘플 문서 수: {len(sample_docs)}")
for doc in sample_docs:
    print(f"  - {doc.metadata.get('title')}")

In [ ]:
# 임베딩 모델 초기화 (OpenAI 또는 HuggingFace)
# OpenAI API 키가 있으면 OpenAI 사용, 없으면 HuggingFace 사용
provider = "openai" if os.getenv("OPENAI_API_KEY") else "huggingface"
embedding_manager = EmbeddingManager(provider=provider)
embeddings = embedding_manager.embeddings

print(f"임베딩 모델 준비 완료 (provider: {provider})")

In [ ]:
# PostgreSQL + pgvector 벡터 스토어 생성
vector_store = PostgresVectorStore(
    embeddings=embeddings,
    collection_name="sample_docs"
)

# 기존 컬렉션 초기화 (선택사항)
try:
    vector_store.clear()
except Exception:
    pass  # 컬렉션이 없으면 무시

# 문서 추가
vector_store.from_documents(sample_docs)

# 통계 확인
stats = vector_store.get_collection_stats()
print(f"컬렉션 통계: {stats}")

In [ ]:
# 유사도 검색 테스트
query = "Spring Security에서 인증을 어떻게 구현하나요?"
results = vector_store.similarity_search_with_score(query, k=2)

print(f"질문: {query}\n")
for doc, score in results:
    print(f"[점수: {score:.4f}] {doc.metadata.get('title')}")
    print(f"  {doc.page_content[:100]}...\n")

---
## 3. RAG 체인 구성

In [ ]:
# Anthropic Claude LLM 초기화
llm_adapter = LLMAdapter(provider="anthropic", temperature=0)
llm = llm_adapter.llm

# Retriever 생성
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# RAG 체인 구성
rag_chain = RAGChain(llm=llm, retriever=retriever)

print("RAG 체인 준비 완료 (Anthropic Claude)")

In [ ]:
# RAG 질의응답 테스트
questions = [
    "Spring Boot에서 JPA 설정은 어떻게 하나요?",
    "REST API URL은 어떻게 설계해야 하나요?",
    "Spring Security에서 특정 URL을 공개하려면 어떻게 하나요?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"질문: {q}")
    print(f"{'='*60}")
    answer = rag_chain.invoke(q)
    print(f"답변: {answer}")

In [ ]:
# 문서에 없는 내용 질문 (Hallucination 방지 테스트)
question = "Django에서 모델을 어떻게 만드나요?"
print(f"질문: {question}")
answer = rag_chain.invoke(question)
print(f"답변: {answer}")

In [ ]:
# 소스와 함께 답변 받기
question = "HTTP 상태 코드 201은 언제 사용하나요?"
result = rag_chain.invoke_with_sources(question)

print(f"질문: {question}\n")
print(f"답변: {result['answer']}\n")
print("참조 문서:")
for i, src in enumerate(result['sources'], 1):
    print(f"  {i}. {src['title']}")

---
## 4. 노션 데이터베이스 연동

실제 노션 데이터베이스에서 문서를 로드합니다.

**주의**: 아래 코드를 실행하려면 `.env`에 `NOTION_API_KEY`가 설정되어 있어야 합니다.

In [ ]:
# 노션 API 키 확인
notion_key = os.getenv("NOTION_API_KEY")
if not notion_key or notion_key == "your-notion-integration-token-here":
    print("⚠️ NOTION_API_KEY가 설정되지 않았습니다.")
    print("   .env 파일에 실제 노션 Integration 토큰을 설정하세요.")
    USE_NOTION = False
else:
    print("✓ NOTION_API_KEY 설정됨")
    USE_NOTION = True

In [ ]:
# 노션 연동 시 아래 코드 실행
if USE_NOTION:
    from src.loaders.notion_loader import NotionDocumentLoader

    # 데이터베이스 ID 설정 (노션 URL에서 확인)
    DATABASE_ID = os.getenv("NOTION_DATABASE_ID", "your-database-id")

    # 노션 로더 초기화
    notion_loader = NotionDocumentLoader(
        chunk_size=500,
        chunk_overlap=100
    )

    # 문서 로드 및 분할
    notion_docs = notion_loader.load_and_split(DATABASE_ID)
    print(f"로드된 청크 수: {len(notion_docs)}")
else:
    print("노션 연동을 건너뜁니다. 샘플 문서를 계속 사용합니다.")

In [ ]:
# 노션 문서로 벡터 스토어 구축
if USE_NOTION and 'notion_docs' in dir():
    notion_vector_store = PostgresVectorStore(
        embeddings=embeddings,
        collection_name="notion_docs"
    )
    
    # 기존 컬렉션 초기화
    try:
        notion_vector_store.clear()
    except Exception:
        pass
    
    notion_vector_store.from_documents(notion_docs)

    # 노션 RAG 체인 구성
    notion_retriever = notion_vector_store.as_retriever(search_kwargs={"k": 4})
    notion_rag = RAGChain(llm=llm, retriever=notion_retriever)

    print("노션 RAG 시스템 준비 완료 (PostgreSQL + pgvector)")

---
## 5. 대화형 RAG (Conversational RAG)

In [ ]:
from src.chains.rag_chain import ConversationalRAGChain

# 대화형 RAG 체인 생성
conversational_rag = ConversationalRAGChain(
    llm=llm,
    retriever=retriever,
    max_history=5
)

print("대화형 RAG 준비 완료")

In [ ]:
# 대화 테스트
conversations = [
    "Spring Boot에서 JPA를 사용하려면 어떻게 해야 하나요?",
    "방금 말한 설정에서 show-sql 옵션은 무엇인가요?",
    "그럼 H2 대신 MySQL을 사용하려면 어떻게 바꾸나요?"
]

for q in conversations:
    print(f"\n사용자: {q}")
    answer = conversational_rag.invoke(q)
    print(f"AI: {answer}")

In [ ]:
# 대화 히스토리 확인
print("현재 대화 히스토리:")
for i, entry in enumerate(conversational_rag.chat_history, 1):
    print(f"\n[대화 {i}]")
    print(f"Q: {entry['question'][:50]}...")
    print(f"A: {entry['answer'][:50]}...")

In [ ]:
# 히스토리 초기화
conversational_rag.clear_history()

---
## 6. 커스텀 프롬프트

특정 도메인이나 스타일에 맞는 커스텀 프롬프트를 적용할 수 있습니다.

In [ ]:
# 개발 문서 특화 프롬프트
DEV_DOC_PROMPT = """당신은 개발 문서를 기반으로 질문에 답변하는 AI 어시스턴트입니다.

다음 가이드라인을 따르세요:
1. 제공된 컨텍스트만을 기반으로 답변하세요.
2. 코드 예시가 있다면 코드 블록으로 표시하세요.
3. 답변은 간결하고 실용적으로 작성하세요.
4. 컨텍스트에 없는 내용은 "해당 정보가 문서에 없습니다."라고 답변하세요.

컨텍스트:
{context}

질문: {question}

답변:"""

# 커스텀 프롬프트로 RAG 체인 생성
custom_rag = RAGChain(
    llm=llm,
    retriever=retriever,
    prompt_template=DEV_DOC_PROMPT
)

# 테스트
question = "REST API에서 리소스 생성 성공 시 어떤 상태 코드를 반환해야 하나요?"
print(f"질문: {question}")
print(f"\n답변: {custom_rag.invoke(question)}")

---
## 정리

이 노트북에서 학습한 내용:

1. **노션 Integration 설정** - API 토큰 발급 및 권한 부여
2. **문서 로드** - NotionDBLoader를 통한 데이터베이스 로드
3. **벡터 스토어** - PostgreSQL + pgvector를 활용한 문서 임베딩 저장
4. **RAG 체인** - Retriever + Anthropic Claude 연결
5. **대화형 RAG** - 히스토리 유지 기능 추가
6. **커스텀 프롬프트** - 도메인 특화 프롬프트 적용

다음 단계:
- Phase 4: 로컬 LLM(Qwen)으로 전환하여 API 비용 절감
- Phase 5: LoRA 튜닝으로 도메인 특화 성능 향상

In [ ]:
# 리소스 정리 (선택사항)
# vector_store.delete_collection()